# Promega Dataset: Descriptive Statistics

Exploratory summary of the data produced by the Promega pre-processing pipeline, based on `all_data.json`. Covers record counts, data availability by modality, and class distributions across all timepoints.

## Contents

| Section | Description |
|---|---|
| **Organoids** | Total unique organoids; count per day |
| **Records** | Total records (one per organoid per day) |
| **Images** | Records with a resized image; count per day |
| **Masks** | Records with a manual mask or predicted mask; count per day |
| **Labels** | Records with an Acceptable/Not Acceptable label; distribution overall and by day |
| **Metabolites** | Records with metabolite assay data (GlucoseGlo, LactateGlo, etc.); count per day |
| **Surveys** | Records with human evaluation data; vote distribution overall and by day |

## dependencies

In [26]:
import collections
import importlib
import json
import pathlib

import pandas as pd

import stats_utils
importlib.reload(stats_utils)

<module 'stats_utils' from '/Users/ntebaldi/Documents/workspace/dsi/promega/workspace/2025-promega-mini-test/notebooks/stats_utils.py'>

## load JSON data

In [27]:
JSON_FILE = pathlib.Path("/Users/ntebaldi/Documents/workspace/dsi/promega/data/identifiers/all_data.json")
json_data = stats_utils.load_json(JSON_FILE)

stats_dict = stats_utils.initialize_stats_dict()

## number of organoids

In [28]:
# unique number of organoids
stats_dict["num_organoids"] = len({ record["organoid_id"] for record in json_data.values() })
print(f"Number of organoids: {stats_dict['num_organoids']}")

Number of organoids: 475


In [29]:
stats_dict["organoid_distr"] = stats_utils.get_distribution_by_day(json_data, lambda r: r["organoid_id"])
stats_utils.to_dataframe(stats_dict["organoid_distr"], key_col="Day", value_col="Organoids", title="Organoids by Day")

,Day,Organoids
0,3.000000,475
1,6.000000,475
2,8.000000,473
3,10.000000,470
4,13.000000,467
5,15.000000,462
6,17.000000,460
7,20.500000,461
8,24.000000,466
9,28.000000,460


## number of records

In [30]:
# number of records for each organoid over all days
stats_dict["num_records"] = len(json_data.values())
print(f"Number of records: {stats_dict['num_records']}")

Number of records: 5168


## number of images

In [31]:
# unique number of images
stats_dict["num_images"] = len({ record.get("images", {}).get("img_path") for record in json_data.values() })
print(f"Number of image records: {stats_dict['num_images']}")

Number of image records: 5168


In [32]:
stats_dict["image_distr"] = stats_utils.get_distribution_by_day(json_data, lambda r: r.get("images", {}).get("img_path"))
stats_utils.to_dataframe(stats_dict["image_distr"], key_col="Day", value_col="Images", title="Images by Day")

,Day,Images
0,3.000000,475
1,6.000000,475
2,8.000000,473
3,10.000000,470
4,13.000000,467
5,15.000000,462
6,17.000000,461
7,20.500000,466
8,24.000000,475
9,28.000000,474


## number of masks

In [33]:
stats_dict["num_masks_manual"] = sum(
    1 for record in json_data.values()
    if record.get("images", {}).get("manual_mask_path_orginal") is not None
)
print(f"Number of manual mask records: {stats_dict['num_masks_manual']}")

stats_dict["masks_manual_distr"] = stats_utils.get_distribution_by_day(json_data, lambda r: r.get("images", {}).get("manual_mask_path_orginal"))
stats_utils.to_dataframe(stats_dict["masks_manual_distr"], key_col="Day", value_col="Manual Masks", title="Manual Masks by Day")

Number of manual mask records: 2090


,Day,Manual Masks
0,3.000000,221
1,6.000000,210
2,8.000000,122
3,10.000000,162
4,13.000000,183
5,15.000000,170
6,17.000000,200
7,20.500000,184
8,24.000000,206
9,28.000000,219


In [34]:
stats_dict["num_masks_predicted"] = sum(
    1 for record in json_data.values()
    if record.get("images", {}).get("mask_path") is not None
)
print(f"Number of predicted mask records: {stats_dict['num_masks_predicted']}")

stats_dict["masks_predicted_distr"] = stats_utils.get_distribution_by_day(json_data, lambda r: r.get("images", {}).get("mask_path"))
stats_utils.to_dataframe(stats_dict["masks_predicted_distr"], key_col="Day", value_col="Predicted Masks", title="Predicted Masks by Day")

Number of predicted mask records: 5168


,Day,Predicted Masks
0,3.000000,475
1,6.000000,475
2,8.000000,473
3,10.000000,470
4,13.000000,467
5,15.000000,462
6,17.000000,461
7,20.500000,466
8,24.000000,475
9,28.000000,474


## number of labels

In [35]:
# unique number of labels
stats_dict["num_labels"] = sum(
    1 for record in json_data.values()
    if record.get("label", {}).get("value") is not None
)
print(f"Number of labels records: {stats_dict['num_labels']}")

Number of labels records: 2931


In [36]:
stats_dict["label_distr"], stats_dict["label_day_distr"] = stats_utils.count_by_value_and_day(
    json_data,
    lambda r: [r["label"]["value"]] if r.get("label", {}).get("value") is not None else []
)

display(stats_utils.to_dataframe(stats_dict["label_distr"], key_col="Label", title="Label Distribution"))

rows = [{"Day": day, **counts} for day, counts in sorted(stats_dict["label_day_distr"].items())]
pd.DataFrame(rows).style.set_caption("Labels by Day")

,Label,Count
0,Acceptable,2125
1,Not Acceptable,806


,Day,Acceptable,Not Acceptable
0,3.000000,192,73
1,6.000000,192,73
2,8.000000,192,73
3,10.000000,192,73
4,13.000000,192,73
5,15.000000,192,73
6,17.000000,192,73
7,20.500000,195,73
8,24.000000,194,74
9,28.000000,196,74


## number of metabolites

In [37]:
# number of metabolites
stats_dict["num_metabolites"] = sum(
    1 for record in json_data.values()
    if record.get("metabolite", {})
)
print(f"Number of metabolite records: {stats_dict['num_metabolites']}")

Number of metabolite records: 4154


In [38]:
stats_dict["metabolite_distr"], stats_dict["metabolite_day_distr"] = stats_utils.count_by_value_and_day(
    json_data,
    lambda r: list(r.get("metabolite", {}).keys())
)

display(stats_utils.to_dataframe(stats_dict["metabolite_distr"], key_col="Metabolite", title="Metabolite Distribution"))

rows = [{"Day": day, **counts} for day, counts in sorted(stats_dict["metabolite_day_distr"].items())]
pd.DataFrame(rows).style.set_caption("Metabolites by Day")

,Metabolite,Count
0,BCAAGlo,4154
1,GlucoseGlo,4154
2,GlutamateGlo,4154
3,LactateGlo,4154
4,MalateGlo,4154
5,PyruvateGlo,4154


,Day,GlucoseGlo,GlutamateGlo,MalateGlo,BCAAGlo,LactateGlo,PyruvateGlo
0,3.000000,379,379,379,379,379,379
1,6.000000,379,379,379,379,379,379
2,8.000000,378,378,378,378,378,378
3,10.000000,377,377,377,377,377,377
4,13.000000,374,374,374,374,374,374
5,15.000000,371,371,371,371,371,371
6,17.000000,371,371,371,371,371,371
7,20.500000,376,376,376,376,376,376
8,24.000000,385,385,385,385,385,385
9,28.000000,384,384,384,384,384,384


## number of surveys

In [39]:
# number of surveys
stats_dict["num_surveys"] = sum(
    1 for record in json_data.values()
    if record.get("survey", {})
)
print(f"Number of survey records: {stats_dict['num_surveys']}")

Number of survey records: 393


In [40]:
survey_by_day = collections.Counter(
    record.get("day", {}).get("number")
    for record in json_data.values()
    if record.get("survey", {}).get("evaluations")
)
stats_dict["survey_day_distr"] = dict(survey_by_day)

stats_dict["survey_vote_distr"], stats_dict["survey_votes_by_day"] = stats_utils.count_by_value_and_day(
    json_data,
    lambda r: [
        e["evaluation"]
        for e in r.get("survey", {}).get("evaluations", [])
        if e.get("evaluation") is not None
    ]
)

display(stats_utils.to_dataframe(stats_dict["survey_day_distr"], key_col="Day", value_col="Surveys", title="Surveys by Day"))
display(stats_utils.to_dataframe(stats_dict["survey_vote_distr"], key_col="Vote", title="Vote Distribution"))

rows = [{"Day": day, **counts} for day, counts in sorted(stats_dict["survey_votes_by_day"].items())]
pd.DataFrame(rows).style.set_caption("Survey Votes by Day")

,Day,Surveys
0,28.000000,24
1,30.000000,369


,Vote,Count
0,Acceptable,1356
1,Not Acceptable,749


,Day,Acceptable,Not Acceptable
0,28.000000,77,43
1,30.000000,1279,706


## save results

In [41]:
OUTPUT_FILE = pathlib.Path("promega_stats.json")
stats_utils.save_results(stats_dict, OUTPUT_FILE)

Results saved to promega_stats.json
